# 12. Stomach mct

Part of the **[Fig. 6 chapter](fig6.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import cooler
import anndata
import scanpy as sc
import scanpy.external as sce
from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import MCDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# mpl.use('agg')
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")
from scipy.stats import zscore


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'
rna_dir = f'{indir}scRNA/Stomach/Nowicki-Osuch2023/'
mct_dir = f'{indir}mCT/'
outdir = f'{indir}analysis/stomach_mct/'


In [ ]:
rna_adata = anndata.read_h5ad(f'{rna_dir}Stomach_Nowicki-Osuch2023_ds.h5ad')
rna_adata


In [ ]:
ds = 1
coord_base = 'tsne'

fig, ax = plt.subplots(figsize=(8, 4), dpi=300, constrained_layout=True)

_ = categorical_scatter(data=rna_adata.obs,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'celltype',
                        text_anno=f'celltype',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )


In [ ]:
rna_adata = anndata.AnnData(rna_adata.raw.X, obs=rna_adata.obs, var=rna_adata.raw.var, obsm=rna_adata.obsm)
sc.pp.filter_genes(rna_adata, min_cells=10)
rna_adata.raw = rna_adata.copy()
sc.pp.normalize_total(rna_adata, target_sum=rna_adata.obs['total_counts'].median())
sc.pp.log1p(rna_adata)


In [ ]:
selc = {
    'Tcell': ['Activated_CD4', 'Activated_CD8', 'gdT', 'NK-cells_Gastric', 'Th1_Th17_TReg', 'Tmem'], 
    'Bmem': ['Bmem'], 
    'Bnaive': ['Naïve', 'Naïve_FCMR'], 
    'Others': ['Contamination', 'Erythrocyte'], 
    'Endocri': ['Enteroendocrine_CHGA', 'Enteroendocrine_GHRL'],
    'Plasma': ['IgA1_IGKappa', 'IgA1_IGLambda', 'IgG_IGM_IGKappa'],
    'Macro': ['Macrophages', 'cDC1'], 
    'Mast': ['Mast_Cells', 'Mast_Cells_Activated'], 
    'Epithl': ['Neck-Cells', 'Intestinal_Undifferentiated', 'Foveolar_Differentiated', 'Foveolar_Intermediate', 'Chief', 'Parietal'],
    'Fibro': ['S1', 'S2', 'S3'], 
    'Endo': ['Arterial', 'Arterial capillary', 'Venous']
}
ct2mt = {x:xx for xx in selc for x in selc[xx]}
rna_adata.obs['majortype'] = rna_adata.obs['celltype'].astype(str).map(ct2mt)
rna_adata.obs['majortype'].value_counts()

In [ ]:
rna_adata.var = rna_adata.var.reset_index().set_index('feature_name')


In [ ]:
sc.tl.rank_genes_groups(rna_adata, "majortype", use_raw=False, method="wilcoxon")


In [ ]:
result = rna_adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names
marker = pd.DataFrame(
    {
        f"{group}_{key[:1]}": result[key][group]
        for group in groups
        for key in ["names"]
    }
).head(5).T


In [ ]:
marker

In [ ]:
mct_adata = anndata.read_h5ad(f'{mct_dir}entex_1_rna.h5ad')
mct_adata

In [ ]:
mct_adata.obs['Donor'] = mct_adata.obs.index.str.split('_').str[1]
mct_adata.obs['Donor'].value_counts()

In [ ]:
from scipy.sparse import csr_matrix
mct_adata.X = csr_matrix(mct_adata.X)


In [ ]:
mct_adata.X.data

In [ ]:
fig, axes = plt.subplots(2, 1, dpi=300)
ax = axes[0]
sns.histplot(data=mct_adata.obs, x='UniqueAlignFinalReads', hue='Donor', log_scale=True, bins=100, ax=ax)
ax = axes[1]
sns.histplot(data=mct_adata.obs, x='mCCCFrac', hue='Donor', bins=100, ax=ax)
fig.tight_layout()


In [ ]:
# mct_adata.obs['n_genes'] = (mct_adata.X > 0).sum(axis=1).A1
mct_adata.obs['n_counts'] = mct_adata.X.sum(axis=1).A1

In [ ]:
sc.pp.filter_cells(mct_adata, min_genes=200)
sc.pp.filter_genes(mct_adata, min_cells=10)


In [ ]:
fig, axes = plt.subplots(2, 1, dpi=300)
ax = axes[0]
sns.histplot(data=mct_adata.obs, x='n_genes', hue='Donor', bins=100, ax=ax)
ax = axes[1]
sns.histplot(data=mct_adata.obs, x='n_counts', hue='Donor', log_scale=True, bins=100, ax=ax)
fig.tight_layout()


In [ ]:
mct_adata.raw = mct_adata.copy()
sc.pp.highly_variable_genes(mct_adata, n_top_genes=3000, flavor='seurat_v3', batch_key='Donor')
sc.pp.normalize_total(mct_adata, target_sum=mct_adata.obs['n_counts'].median())
sc.pp.log1p(mct_adata)
mct_adata = mct_adata[:, mct_adata.var['highly_variable']].copy()
sc.pp.scale(mct_adata, max_value=10)


In [ ]:
sc.tl.pca(mct_adata, svd_solver='arpack', random_state=0, n_comps=50)
mct_adata.obsm['RNA_pca'] = mct_adata.obsm['X_pca'].copy()
significant_pc_test(mct_adata, p_cutoff=0.05, update=False, obsm='X_pca')


In [ ]:
npc = 20
mct_adata.obsm['X_pca'] = normalize(mct_adata.obsm['RNA_pca'][:, :npc], axis=1)
tsne(mct_adata, obsm='X_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mct_adata.obsm[f'RNA_pc{npc}_tsne'] = mct_adata.obsm['X_tsne'].copy()


In [ ]:
sce.pp.harmony_integrate(mct_adata, 'Donor', basis='X_pca', max_iter_harmony=30, random_state=0)
tsne(mct_adata, obsm='X_pca_harmony', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mct_adata.obsm[f'RNA_pc{npc}hm_tsne'] = mct_adata.obsm['X_tsne'].copy()


In [ ]:
mct_adata.obs.loc[mcad.obs.index, f'5kCG_leiden'] = mcad.obs['5kCG_u17hm_leiden'].values.copy()

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i,mode in enumerate(['', 'hm']):
    mct_adata.obsm[f'X_{coord_base}'] = mct_adata.obsm[f'RNA_pc{npc}{mode}_tsne'].copy()
    dump_embedding(mct_adata, coord_base)
    tmp = mct_adata.obs.copy()
    ax = axes[0,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            text_anno='Donor',
                            labelsize=8,
                            s=ds,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            show_legend=True
                            )
    ax = axes[1,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='5kCG_leiden',
                            text_anno='5kCG_leiden',
                            labelsize=8,
                            s=ds,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            # show_legend=True
                            )


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i,mode in enumerate(['', 'hm']):
    mct_adata.obsm[f'X_{coord_base}'] = mct_adata.obsm[f'RNA_pc{npc}{mode}_tsne'].copy()
    dump_embedding(mct_adata, coord_base)
    tmp = mct_adata.obs.copy()
    ax = axes[0,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            text_anno='Donor',
                            labelsize=8,
                            s=ds,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            show_legend=True
                            )
    ax = axes[1,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='5kCG_leiden',
                            text_anno='5kCG_leiden',
                            labelsize=8,
                            s=ds,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            # show_legend=True
                            )


In [ ]:
mct_adata.write_h5ad(f'{outdir}mCT_raw.h5ad')

In [ ]:
print((mct_adata.obs['UniqueAlignFinalReads']>500000).sum(), 
      (mct_adata.obs['mCCCFrac']<0.1).sum(), 
      (mct_adata.obs['n_genes']>200).sum(), 
      mct_adata.shape[0])


In [ ]:
selc = mct_adata.obs.index[(mct_adata.obs['UniqueAlignFinalReads']>500000) & (mct_adata.obs['mCCCFrac']<0.1)]
print(len(selc))


In [ ]:
mcad = anndata.read_h5ad(f'{outdir}5kCG_embed.h5ad')
mcad

In [ ]:
selc = [[0,1,3,4,14], [2,5], [6], [7], [8], [9], [10], [11], [12], [13], [15]]
clusterdict = {str(xx):str(i) for i,x in enumerate(selc) for xx in x}

mcad.obs['cluster'] = mcad.obs['5kCG_u17hm_leiden'].map(clusterdict).astype(str)
mcad.obs['cluster'].value_counts()


In [ ]:
mct_adata = mct_adata[mcad.obs.index].copy()

In [ ]:
data = pd.DataFrame(mct_adata.X, index=mct_adata.obs.index, columns=mct_adata.var.index)
data[['cluster', 'Donor']] = mcad.obs[['cluster', 'Donor']].copy()
data = data.groupby(['cluster', 'Donor']).sum()


In [ ]:
design = data.index.to_frame().astype(str)
design.index = design['cluster'].astype(str) + '_' + design['Donor'].astype(str)
data.index = design.index.copy()

In [ ]:
outdir

In [ ]:
data.to_hdf(f'{outdir}mc_cluster_expr.hdf', key='data')
design.to_hdf(f'{outdir}mc_cluster_expr.hdf', key='design')


In [ ]:
data = pd.read_hdf(f'{outdir}mc_cluster_expr.hdf', key='data')
design = pd.read_hdf(f'{outdir}mc_cluster_expr.hdf', key='design')
print(data.shape, design.shape)

In [ ]:
import itertools

params = []
ct_list = design['cluster'].unique()
for ct1,ct2 in itertools.combinations(ct_list, 2):
    # if ((designtmp['subclass']==ct1).sum() > 1) and ((designtmp['subclass']==ct2).sum() > 1):
    if (design['cluster'].isin([ct1, ct2]).sum() > 2):
        params.append([ct1, ct2])

print(len(params))


In [ ]:
dar_dir = f'{outdir}DEG'
os.makedirs(dar_dir, exist_ok=True)
batch_size = 5
batch = []
for i in range(0, len(params), batch_size):
    batch.append([i, i+batch_size])

batch[-1][-1] = len(params)
pd.DataFrame(batch).to_csv(f'{dar_dir}/diff_expr_batch.txt', index=False, header=False, sep=' ')
pd.DataFrame(params).to_csv(f'{dar_dir}/diff_expr_group.txt', index=False, header=False, sep=' ')


In [ ]:
params = pd.read_csv(f'{outdir}DEG/diff_expr_group.txt', sep=' ', header=None, index_col=None).values
peak_list = pd.Index([])
for ct1, ct2 in params:
    # selc = (design['age_group']==age_group) & design['subclass'].isin([ct1, ct2])
    group_name = f'{ct1}-{ct2}'.replace(' ', '_').replace('/', '-')
    stats = pd.read_hdf(f'{outdir}DEG/results/{group_name}_stats.hdf', key='data')
    peak_list = peak_list.union(stats.index[stats['padj']<0.01])

print(len(peak_list))


In [ ]:
# type_palette = {'Control':'#d95f02', 'Sporadic':'#1b9e77', 'C9ORF72':'#7570b3'}
# sex_palette = {'Male':'#a65628', 'Female':'#f781bf', 'Unknown':'#999999'}

datatmp = data / data.sum(axis=1).values[:,None] * 1e6
ens2gene = mct_adata.var['gene_name'].to_dict()
data_tmp = datatmp.loc[:, peak_list].T
data_tmp.index = data_tmp.index.map(ens2gene)
# col_colors = [design.loc[selc, 'Type'].map(type_palette), 
#                 design.loc[selc, 'Sex'].map(sex_palette)]
cg = sns.clustermap(data_tmp, cmap='vlag', z_score=0, vmin=-2, vmax=2, metric='cosine',
                   # col_colors=col_colors, 
                   xticklabels=1, figsize=(5,4))
cg.ax_row_dendrogram.set_visible(False)
# cg.ax_col_dendrogram.set_title(f'{subclass} ({len(selp)})', fontsize=16)

# handles = [Patch(facecolor=sex_palette[name]) for name in sex_palette] + [Patch(facecolor=type_palette[name]) for name in type_palette]
# labels = list(sex_palette.keys()) + list(type_palette.keys())

# plt.legend(handles, labels, title='Column Categories', # fontsize=8, 
#         bbox_to_anchor=(0, -0.5), loc='upper left', borderaxespad=0.)


In [ ]:
rna_adata = anndata.AnnData(X=rna_adata.raw.X, obs=rna_adata.obs, var=rna_adata.raw.var, obsm=rna_adata.obsm)
rna_adata

In [ ]:
data_rna = pd.DataFrame(rna_adata.X.toarray(), index=rna_adata.obs.index, columns=rna_adata.var.index)
data_rna['cell_type'] = rna_adata.obs['cell_type'].copy()
data_rna = data_rna.groupby('cell_type').sum()


In [ ]:
peak_list = peak_list.str.split('.').str[0]
peak_list = peak_list[peak_list.isin(data_rna.columns)]


In [ ]:
data.columns = data.columns.str.split('.').str[0]

In [ ]:
ens2gene = mct_adata.var['gene_name']
ens2gene.index = ens2gene.index.str.split('.').str[0]
ens2gene = ens2gene.to_dict()


In [ ]:
datatmp = data / data.sum(axis=1).values[:,None] * 1e6
data_tmp = datatmp.loc[:, peak_list].T
data_tmp.index = data_tmp.index.map(ens2gene)
# col_colors = [design.loc[selc, 'Type'].map(type_palette), 
#                 design.loc[selc, 'Sex'].map(sex_palette)]
cg = sns.clustermap(data_tmp, cmap='vlag', z_score=0, vmin=-2, vmax=2, metric='cosine',
                   # col_colors=col_colors, 
                   xticklabels=1, figsize=(5,4))
cg.ax_row_dendrogram.set_visible(False)
rorder = cg.dendrogram_row.reordered_ind.copy()


In [ ]:
from scipy.stats import zscore

datatmp = data_rna / data_rna.sum(axis=1).values[:,None] * 1e6
data_tmp = zscore(datatmp.loc[:, peak_list[rorder]].T, axis=1)
data_tmp.index = data_tmp.index.map(ens2gene)
# col_colors = [design.loc[selc, 'Type'].map(type_palette), 
#                 design.loc[selc, 'Sex'].map(sex_palette)]
sns.heatmap(data_tmp, cmap='vlag', vmin=-2, vmax=2, 
                   # col_colors=col_colors, 
                   xticklabels=1)


In [ ]:
mct_adata = anndata.read_h5ad(f'{outdir}mCT_raw.h5ad')
mct_adata

In [ ]:
data = pd.DataFrame(mct_adata.X.toarray(), index=mct_adata.obs.index, columns=mct_adata.var['gene_name'])
data

In [ ]:
marker

In [ ]:
coord_base = 'tsne'
ds = 4

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(4*5,3), dpi=300)
tmp = mcad.obs.copy()
for i,xx in enumerate(marker.loc['Endo_n']):
    ax = axes[i]
    if xx in data.columns:
        _ = continuous_scatter(data=tmp,
                               ax=ax,
                               hue=zscore(data[xx]),
                               coord_base=coord_base,
                               s=ds,
                               labelsize=8,
                               max_points=None,
                               cmap='Oranges',
                               hue_norm=[0,2],
                               scatter_kws={'rasterized':True},
                          )
        ax.set_title(xx)
        